In [ ]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("HF token: ")

In [ ]:
print(bool(os.environ.get("HF_TOKEN")))   # має бути True

In [ ]:

import torch; print(torch.cuda.is_available())

In [ ]:
!git clone https://github.com/Luzefik/storozha-anti-ambush.git

In [ ]:
!ls



In [ ]:
cd storozha-anti-ambush

In [ ]:
ls

In [ ]:
!git checkout feat/project-scaffold
!git log --oneline -1
!pip install -q -e . --no-deps


In [ ]:
!pip install ultralytics
!pip install -q kaggle
!kaggle competitions download -c predicting-assessment-scores-hw1-ucu-ai-2026 -p data
!unzip -q -o data/*.zip -d data
!ls data
!pip install -q pandas numpy seaborn matplotlib scikit-learn xgboost
!pip install seaborn


In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV,
                                     cross_val_predict, cross_validate)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from ultralytics import YOLO

In [ ]:

import subprocess
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/storozha/runs"
Path(RUNS).mkdir(parents=True, exist_ok=True)


def log_run(name, runs=RUNS):
    df = pd.read_csv(f"{runs}/{name}/results.csv")
    df.columns = df.columns.str.strip()
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    last10 = df.tail(10)
    row = {
        "run": name,
        "epochs_run": len(df),
        "best_epoch": int(best["epoch"]),
        "best_mAP50": round(best["metrics/mAP50(B)"], 4),
        "best_recall": round(best["metrics/recall(B)"], 4),
        "last10_mAP50": round(last10["metrics/mAP50(B)"].mean(), 4),
        "git_commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                     capture_output=True, text=True).stdout.strip(),
    }
    log = Path(runs) / "summary.csv"
    pd.DataFrame([row]).to_csv(log, mode="a", header=not log.exists(), index=False)
    print(row)
    return row

In [ ]:
from ultralytics import YOLO

RUN = "yolo26n-640"
model = YOLO("yolo26n.pt")
model.train(data="data/processed/yolo/data.yaml",
            imgsz=640, epochs=100, patience=20, batch=-1, seed=42,
            device=0, cache="ram", workers=2,
            project=RUNS, name=RUN)   # -> /content/drive/MyDrive/storozha/runs/<RUN>
RUN = Path(model.trainer.save_dir).name
log_run(RUN)

In [ ]:
from ultralytics import RTDETR

RUN = "rtdetr-l-640"
model = RTDETR("rtdetr-l.pt")
model.train(data="data/processed/yolo/data.yaml",
            imgsz=640, epochs=100, patience=20, batch=8, seed=42,
            device=0, workers=2,
            project=RUNS, name=RUN)
RUN = Path(model.trainer.save_dir).name
log_run(RUN)

In [ ]:
from ultralytics import YOLO, RTDETR

RUN = "yolo26n-1280"
weights = f"{RUNS}/{RUN}/weights/best.pt"
Model = RTDETR if RUN.startswith("rtdetr") else YOLO
model = Model(weights)

# відео з test-спліту; результат (відео з рамками) — лише в приватну папку на Drive
video = "data/raw/Z-FPV-v1/videos/video-24.mp4"
results = model.predict(source=video, imgsz=1280, conf=0.25, device=0,
                        stream=True, save=True,
                        project="/content/drive/MyDrive/storozha/predictions",
                        name=f"{RUN}_video-24")
n_frames = n_det = 0
for r in results:          # зі stream=True обробка йде лише під час перебору
    n_frames += 1
    n_det += len(r.boxes) > 0
print(f"кадрів: {n_frames}, з детекцією: {n_det}")

In [ ]:

from pathlib import Path
import yaml


yolo = Path("data/processed/yolo")
m = pd.read_csv("config/splits/video_split_manifest.csv")
pool = m[m["split"] != "test"]

K = 4
order = pd.concat([pool[pool.is_night], pool[~pool.is_night].sort_values("n_frames", ascending=False)])
folds = [[] for _ in range(K)]
for i, v in enumerate(order["video"]):
    folds[i % K].append(v)

def lines(videos):
    return [f"./images/{v}/{p.name}" for v in videos
            for p in sorted((yolo / "images" / v).glob("frame_*.png"))]

for k, val_videos in enumerate(folds):
    train_videos = [v for v in pool["video"] if v not in val_videos]
    (yolo / f"fold{k}_train.txt").write_text("\n".join(lines(train_videos)) + "\n")
    (yolo / f"fold{k}_val.txt").write_text("\n".join(lines(val_videos)) + "\n")
    (yolo / f"fold{k}.yaml").write_text(yaml.safe_dump(
        {"train": f"fold{k}_train.txt", "val": f"fold{k}_val.txt",
         "names": {0: "waiter_fpv_drone"}}, sort_keys=False))
    print(k, "val:", val_videos)

In [ ]:
import numpy as np
from ultralytics import YOLO

CV_NAME = "cv-yolo11s-1240"                 # назва всього експерименту з крос-валідацією
CV_DIR = f"{RUNS}/{CV_NAME}"
fold_maps = []
for k in range(K):
    model = YOLO("yolo11s.pt")
    model.train(data=f"data/processed/yolo/fold{k}.yaml", imgsz=1240, epochs=50,
                patience=15, batch=-1, seed=42, device=0, cache="ram", workers=2,
                project=CV_DIR, name=f"fold{k}")
    fold_maps.append(model.metrics.box.map50)          # mAP@0.5 на val цього фолду
    log_run(Path(model.trainer.save_dir).name, runs=CV_DIR)   # summary.csv усередині CV_DIR

print(f"mAP50 = {np.mean(fold_maps):.3f} ± {np.std(fold_maps):.3f}", fold_maps)
pd.DataFrame({"fold": range(K), "mAP50": fold_maps}).to_csv(f"{CV_DIR}/cv_results.csv", index=False)